In [3]:
import json
import warnings
import numpy as np
import pandas as pd
import optuna
import torch

warnings.filterwarnings("ignore")

# SDV: Giữ lại để đánh giá Quality Score cho Optuna (đảm bảo tính nhất quán metric)
from sdv.metadata import SingleTableMetadata
from sdv.evaluation.single_table import evaluate_quality

# Synthcity: Lõi TabDDPM
from synthcity.plugins import Plugins
from synthcity.plugins.core.dataloader import GenericDataLoader

# =========================
# 1) Config & Hyperparameters
# =========================
SEED = 121
np.random.seed(SEED)
torch.manual_seed(SEED)

TRAIN_PATH = r"C:\Users\NFU_Power_Lab\Desktop\DAT_ResearchArea\BestAwardAI2026\random-split\wpt_train_random.csv"
OUT_AUG_PATH = "wpt_train_random_aug_tabddpm.csv"          
OUT_SYN_ONLY_PATH = "wpt_train_random_syn_only_tabddpm.csv" 
OPTUNA_PARAMS_PATH = "best_tabddpm_params.json"             

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Balancing Policy
MAX_SYN_MULTIPLIER = 2.0 
TARGET_MINIMUM = 50      
SYN_PER_TYPE_CAP = 300    

# Optuna Config
OPTUNA_TRIALS = 20  # TabDDPM chạy khá nặng, 20 trials là đủ
FINAL_EPOCHS = 2000 # TabDDPM dùng số vòng lặp (iterations) cao hơn GAN

# =========================
# 2) Load & Preprocess Data
# =========================
df = pd.read_csv(TRAIN_PATH)
expected = {"type", "load", "efficiency"}
if expected - set(df.columns):
    raise ValueError(f"Missing columns: {expected - set(df.columns)}")

df["type"] = df["type"].astype(str)
df["load"] = pd.to_numeric(df["load"], errors="coerce")
df["efficiency"] = pd.to_numeric(df["efficiency"], errors="coerce")
df = df.dropna(subset=["type", "load", "efficiency"]).reset_index(drop=True)

print("Train shape:", df.shape)

val_ratio = 0.2
val_n = int(len(df) * val_ratio)
df_train = df.iloc[val_n:].reset_index(drop=True)
df_val = df.iloc[:val_n].reset_index(drop=True)

# DataLoader cho Synthcity
loader_train = GenericDataLoader(df_train)

# Metadata cho SDV (để đánh giá Optuna)
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(df)
metadata.update_column(column_name='type', sdtype='categorical')
metadata.update_column(column_name='load', sdtype='numerical')
metadata.update_column(column_name='efficiency', sdtype='numerical')

# =========================
# 3) Setup Optuna Objective Function cho TabDDPM
# =========================
def objective(trial):
    n_iter = trial.suggest_int("n_iter", 500, 1500, step=500)
    batch_size = trial.suggest_categorical("batch_size", [128, 256])
    lr = trial.suggest_float("lr", 1e-4, 5e-3, log=True)
    
    # Khởi tạo mô hình TabDDPM
    synthesizer = Plugins().get(
        "ddpm",
        n_iter=n_iter,
        batch_size=batch_size,
        lr=lr,
        device=DEVICE
    )
    
    # Huấn luyện
    synthesizer.fit(loader_train)
    
    try:
        # Generate data
        syn_val = synthesizer.generate(len(df_val)).dataframe()
        
        # Đánh giá bằng SDV để lấy Quality Score
        quality_report = evaluate_quality(
            real_data=df_val,
            synthetic_data=syn_val,
            metadata=metadata,
            verbose=False
        )
        score = quality_report.get_score()
    except Exception as e:
        score = 0.0

    return score

# =========================
# 4) Run Optuna Optimization
# =========================
print(f"\n=== Bắt đầu tìm kiếm Optuna ({OPTUNA_TRIALS} Trials) ===")
study = optuna.create_study(direction="maximize") 
study.optimize(objective, n_trials=OPTUNA_TRIALS)

print("\n=== Optuna Hoàn Thành ===")
best_params = study.best_params
print(f"Điểm Quality Score cao nhất: {study.best_value:.4f}")
print(f"Các tham số tốt nhất: {best_params}")

with open(OPTUNA_PARAMS_PATH, 'w') as f:
    json.dump(best_params, f, indent=4)

# =========================
# 5) Train Final TabDDPM Model
# =========================
print(f"\n=== Bắt đầu huấn luyện mô hình TabDDPM cuối cùng ===")
final_synthesizer = Plugins().get(
    "ddpm",
    n_iter=FINAL_EPOCHS, # Dùng số lần lặp lớn nhất cho mô hình chốt
    batch_size=best_params["batch_size"],
    lr=best_params["lr"],
    device=DEVICE
)

# Train trên toàn bộ dữ liệu
final_loader = GenericDataLoader(df)
final_synthesizer.fit(final_loader)
print("Huấn luyện Diffusion thành công!")

# =========================
# 6) Generate Synthetic Data (Over-generation & Filtering)
# =========================
counts = df["type"].value_counts().sort_index()
print("\nCounts by type (before):\n", counts)

syn_parts = []
eff_min, eff_max = df["efficiency"].min(), df["efficiency"].max()

# Sinh trước 10,000 mẫu để tạo "Bể chứa" dữ liệu
print("\nĐang sinh 10,000 mẫu từ không gian Diffusion để lọc...")
pool_df = final_synthesizer.generate(10000).dataframe()
pool_df["efficiency"] = np.clip(pool_df["efficiency"], eff_min, eff_max)

for t in sorted(df["type"].unique()):
    n_real = int((df["type"] == t).sum())

    if t == 'K' or n_real > 300:
        continue

    desired_target = max(TARGET_MINIMUM, int(counts.median()))
    need = desired_target - n_real
    max_allowed_syn = int(n_real * MAX_SYN_MULTIPLIER)
    need = min(need, max_allowed_syn)
    need = min(need, SYN_PER_TYPE_CAP)
 
    if need <= 0:
        continue

    print(f"Đang trích xuất {need} mẫu giả cho loại vật thể: {t}...")
    
    # Trích xuất từ Bể dữ liệu
    class_pool = pool_df[pool_df["type"] == t]
    
    if len(class_pool) >= need:
        syn_df = class_pool.sample(n=need, random_state=SEED)
    else:
        # Nếu Bể không đủ, liên tục sinh thêm các mẻ 2000 mẫu cho đến khi đủ
        print(f"  -> Không đủ mẫu trong pool, đang khởi động lại Diffusion...")
        collected = [class_pool]
        while sum(len(x) for x in collected) < need:
            extra_df = final_synthesizer.generate(2000).dataframe()
            extra_df["efficiency"] = np.clip(extra_df["efficiency"], eff_min, eff_max)
            collected.append(extra_df[extra_df["type"] == t])
        syn_df = pd.concat(collected).head(need)
    
    syn_df["is_synthetic"] = 1
    syn_parts.append(syn_df)

# =========================
# 7) Merge & Save
# =========================
if syn_parts:
    syn_df_all = pd.concat(syn_parts, ignore_index=True)
else:
    syn_df_all = pd.DataFrame(columns=["type","load","efficiency","is_synthetic"])

real_df = df.copy()
real_df["is_synthetic"] = 0
aug_df = pd.concat([real_df, syn_df_all], ignore_index=True)

before = df["type"].value_counts().sort_index()
after = aug_df["type"].value_counts().sort_index()
report = pd.DataFrame({"before": before, "after": after}).fillna(0).astype(int)

report.to_csv("tabddpm_type_counts_before_after.csv", encoding="utf-8-sig")
aug_df.to_csv(OUT_AUG_PATH, index=False, encoding="utf-8-sig")

if not syn_df_all.empty:
    syn_df_all.to_csv(OUT_SYN_ONLY_PATH, index=False, encoding="utf-8-sig")

print("\nGenerated synthetic rows:", len(syn_df_all))
print("Saved augmented train:", OUT_AUG_PATH)
if not syn_df_all.empty:
    print("Saved synthetic-only data:", OUT_SYN_ONLY_PATH)
print("\nCounts by type (after):\n", after)

# =========================
# 8) Xuất File Thống kê Phân phối
# =========================
stats_before = df.groupby("type")["efficiency"].agg(["mean","std","count"]).rename(columns={"count":"n_before"})
stats_after  = aug_df.groupby("type")["efficiency"].agg(["mean","std","count"]).rename(columns={"count":"n_after"})
stats_merge = stats_before.join(stats_after, lsuffix="_real", rsuffix="_aug")
stats_merge.to_csv("tabddpm_type_stats_before_after.csv", encoding="utf-8-sig")
print("Saved: tabddpm_type_stats_before_after.csv")

[2026-03-11T03:13:06.779387+0800][29188][CRITICAL] load failed: module 'synthcity.plugins.generic.plugin_goggle' has no attribute 'plugin'
[2026-03-11T03:13:06.780511+0800][29188][CRITICAL] load failed: module 'synthcity.plugins.generic.plugin_goggle' has no attribute 'plugin'
[2026-03-11T03:13:06.781049+0800][29188][CRITICAL] module plugin_goggle load failed


Train shape: (1310, 3)

=== Bắt đầu tìm kiếm Optuna (20 Trials) ===


Epoch: 100%|██████████| 1500/1500 [01:53<00:00, 13.24it/s, loss=0.332]
[2026-03-11T03:15:02.164562+0800][29188][CRITICAL] load failed: module 'synthcity.plugins.generic.plugin_goggle' has no attribute 'plugin'
[2026-03-11T03:15:02.164562+0800][29188][CRITICAL] load failed: module 'synthcity.plugins.generic.plugin_goggle' has no attribute 'plugin'
[2026-03-11T03:15:02.165564+0800][29188][CRITICAL] module plugin_goggle load failed
Epoch: 100%|██████████| 500/500 [00:24<00:00, 20.51it/s, loss=0.371]
[2026-03-11T03:15:30.873102+0800][29188][CRITICAL] load failed: module 'synthcity.plugins.generic.plugin_goggle' has no attribute 'plugin'
[2026-03-11T03:15:30.874102+0800][29188][CRITICAL] load failed: module 'synthcity.plugins.generic.plugin_goggle' has no attribute 'plugin'
[2026-03-11T03:15:30.875102+0800][29188][CRITICAL] module plugin_goggle load failed
Epoch: 100%|██████████| 1000/1000 [00:50<00:00, 19.61it/s, loss=0.344]
[2026-03-11T03:16:23.862841+0800][29188][CRITICAL] load failed: m


=== Optuna Hoàn Thành ===
Điểm Quality Score cao nhất: 0.8679
Các tham số tốt nhất: {'n_iter': 500, 'batch_size': 128, 'lr': 0.004668852934839693}

=== Bắt đầu huấn luyện mô hình TabDDPM cuối cùng ===


Epoch: 100%|██████████| 2000/2000 [02:55<00:00, 11.39it/s, loss=0.342]


Huấn luyện Diffusion thành công!

Counts by type (before):
 type
A     22
B     89
C     24
D     95
E     91
F     26
G     95
H    279
I     23
J    185
K    381
Name: count, dtype: int64

Đang sinh 10,000 mẫu từ không gian Diffusion để lọc...
Đang trích xuất 44 mẫu giả cho loại vật thể: A...
Đang trích xuất 2 mẫu giả cho loại vật thể: B...
Đang trích xuất 48 mẫu giả cho loại vật thể: C...
Đang trích xuất 52 mẫu giả cho loại vật thể: F...
Đang trích xuất 46 mẫu giả cho loại vật thể: I...
  -> Không đủ mẫu trong pool, đang khởi động lại Diffusion...

Generated synthetic rows: 192
Saved augmented train: wpt_train_random_aug_tabddpm.csv
Saved synthetic-only data: wpt_train_random_syn_only_tabddpm.csv

Counts by type (after):
 type
A     66
B     91
C     72
D     95
E     91
F     78
G     95
H    279
I     69
J    185
K    381
Name: count, dtype: int64
Saved: tabddpm_type_stats_before_after.csv
